# nb_ingest_cempre_ibge — Ingestão de Dados de Emprego (CEMPRE/RAIS)

**Fonte:** IBGE SIDRA Tabela 3421 (Estatísticas do Cadastro Central de Empresas)
**Variável:** Pessoal ocupado assalariado (708)
**Classificação:** Seção CNAE 2.0 (12762)

In [ ]:
%run ./nb_utils_ibge

## 1. Bronze Layer — Ingestão

In [ ]:
TABLE_CEMPRE = "3421"
VAR_PESSOAL  = "708"
CLASS_CNAE   = {"12762": "allxt"} # Código correto para Seções CNAE na Tabela 3421

print("=== Ingestão CEMPRE/RAIS ===")
df_raw = fetch_sidra_fabric(TABLE_CEMPRE, VAR_PESSOAL, classifications=CLASS_CNAE)

if df_raw:
    save_delta(df_raw, "bronze_ibge_cempre_raw")

## 2. Silver Layer — Padronização

In [ ]:
if df_raw:
    # Padronização robusta usando D-codes
    df_silver = (
        df_raw
        .select(
            col("D1C").cast("int").alias("id_municipio"),
            trim(regexp_replace(col("D1N"), r"\s*\([A-Z]{2}\)$", "")).alias("nome_municipio"),
            col("D2C").cast("int").alias("ano"),
            col("D3C").alias("secao_cnae_cod"),
            col("D3N").alias("secao_cnae"),
            col("V").cast("double").alias("valor")
        )
        .filter(col("secao_cnae_cod").rlike("^[A-Z]$")) # Filtra apenas seções
    )
    
    df_silver = df_silver.withColumn("indicador", lit("pessoal_assalariado"))
    
    save_delta(df_silver, "silver_cempre")
    display(df_silver.limit(10))